# UNIDAD I: Introducción a algoritmos y estructuras de datos
## 🔀 Capitulo 2 DSAP: OOP — Polimorfismo y ADTs
### Programación III - Lic. en Sistemas

![Coding](images/pexels-thirdman-8470810.jpg)

[Foto de Thirdman](https://www.pexels.com/es-es/foto/ordenador-portatil-arquitectura-ordenador-teclado-8470810/)
---

> **Prerequisito:** Notebook 2/3 — Herencia y Encapsulamiento (§2.4–2.5)  
> Este notebook cierra el capítulo con el concepto más poderoso de Python: objetos que cooperan si "hablan el mismo idioma", independientemente de su tipo exacto.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap02/03_Polimorfismo_ADTs_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook serás capaz de:

1. **Explicar** el polimorfismo y el duck typing, y por qué Python los favorece sobre la verificación de tipos
2. **Usar** `isinstance` e `issubclass` correctamente — cuándo sí y cuándo no
3. **Crear** una clase abstracta con `abc.ABC` y `@abstractmethod`
4. **Implementar** el patrón del libro: `Secuencia` ABC con métodos abstractos (`__len__`, `__getitem__`) y métodos concretos heredados (`__contains__`, `__iter__`, `index`, `count`)
5. **Aplicar** el patrón de diseño **Template Method** usando ABCs

## 📑 Tabla de Contenidos

| Sección | Tema | Tiempo estimado |
|---------|------|----------------|
| **§1** | Polimorfismo y Duck Typing | 30 min |
| **§2** | Clases Base Abstractas (ABCs) — `abc` module | 35 min |
| **§3** | Patrón Template Method | 20 min |
| 🧪 | Tests unitarios | 10 min |
| 📋 | Resumen, autoevaluación, referencias | 5 min |
| **Total** | | **~100 min** |

---

### 🦆 La idea central: Duck Typing

> *"Si camina como un pato y grazna como un pato, entonces es un pato."*

Python no verifica el tipo de un objeto — verifica si tiene los **métodos/atributos que necesita**. Esto es el *duck typing*: un objeto es "compatible" con una operación si implementa la interfaz requerida, sin importar su clase concreta.

```text
len(x)       # funciona con list, str, dict, tuple, set — cualquier __len__
for v in x   # funciona con cualquier __iter__ o __getitem__
x + y        # funciona con cualquier __add__
```

Los ABCs (Abstract Base Classes) formalizan este concepto: definen un contrato explícito que las clases concretas deben cumplir.

## ⚙️ Configuración

In [ ]:
import sys
import abc
import unittest
import matplotlib.pyplot as plt

assert sys.version_info >= (3, 8), f"Python 3.8+ requerido. Versión actual: {sys.version}"
print(f"Python {sys.version}")
print("Módulos cargados: sys, abc, unittest, matplotlib")
print("Listo para explorar §2.7 Polimorfismo y ABCs")

---
## 🦆 Sección 1 — Polimorfismo y Duck Typing

### ¿Qué es polimorfismo?

**Polimorfismo** = la misma operación funciona con objetos de distintos tipos.

En Python hay tres formas principales:

| Forma | Mecanismo | Ejemplo |
|-------|-----------|---------|
| **Duck typing** | Verificación de métodos en tiempo de ejecución | `len(x)` funciona con `list`, `str`, `dict`... |
| **Herencia** | Los métodos se resuelven via MRO | `str(obj)` llama `obj.__str__()` correspondiente |
| **Sobrecarga de operadores** | Métodos especiales | `a + b` puede ser suma, concatenación, unión... |

### `isinstance` vs. duck typing

```python
# Estilo Java / verificación de tipo (menos Pythonico)
def procesar(x):
    if isinstance(x, list):
        return x[0]
    elif isinstance(x, str):
        return x[0]

# Estilo Python / duck typing (EAFP: Easier to Ask Forgiveness than Permission)
def procesar(x):
    try:
        return x[0]          # si x soporta __getitem__, funciona
    except (IndexError, TypeError):
        raise ValueError(f"No se puede procesar {type(x).__name__!r}")
```

> 💡 **Regla Pytónica:** preferir duck typing sobre `isinstance` cuando el código sólo necesita que el objeto "sepa hacer algo". Usar `isinstance` cuando genuinamente importa el tipo (por ejemplo, en un comparador o en la API pública de una biblioteca).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Duck typing + sobrecarga de operadores
# ─────────────────────────────────────────────────────────────────────────────

# ── Función polimórfica pura (duck typing) ───────────────────────────────────
def primero_y_ultimo(coleccion):
    """
    Retorna (primero, último) de cualquier secuencia indexable y con len.
    No importa si es list, str, tuple, o clase personalizada.
    """
    if len(coleccion) == 0:
        raise ValueError("La colección está vacía")
    return coleccion[0], coleccion[-1]


print("=== primero_y_ultimo (duck typing) ===")
for obj in [[10, 20, 30], "Python", (1, 2, 3, 4, 5), range(6)]:
    p, u = primero_y_ultimo(obj)
    print(f"  {type(obj).__name__:8s}: primero={p!r}, último={u!r}")


# ── Sobrecarga de operadores: clase Vector ──────────────────────────────────
class Vector:
    """
    Vector n-dimensional con operadores sobrecargados.
    Demuestra __add__, __sub__, __mul__, __rmul__, __len__, __getitem__,
    __iter__, __contains__, __repr__, __eq__.
    """

    def __init__(self, *coords):
        self._coords = tuple(float(c) for c in coords)

    def __len__(self):
        return len(self._coords)

    def __getitem__(self, idx):
        return self._coords[idx]

    def __iter__(self):
        return iter(self._coords)

    def __contains__(self, valor):
        return valor in self._coords

    def __repr__(self):
        coords_str = ', '.join(f'{c:.2f}' for c in self._coords)
        return f"Vector({coords_str})"

    def __eq__(self, other):
        if not isinstance(other, Vector): return NotImplemented
        return self._coords == other._coords

    def __add__(self, other):
        if len(self) != len(other):
            raise ValueError("Los vectores deben tener la misma dimensión")
        return Vector(*(a + b for a, b in zip(self, other)))

    def __sub__(self, other):
        if len(self) != len(other):
            raise ValueError("Los vectores deben tener la misma dimensión")
        return Vector(*(a - b for a, b in zip(self, other)))

    def __mul__(self, escalar):
        """Producto por escalar: v * k"""
        if isinstance(escalar, (int, float)):
            return Vector(*(c * escalar for c in self))
        if isinstance(escalar, Vector):          # producto punto
            if len(self) != len(escalar):
                raise ValueError("Dimensiones distintas para producto punto")
            return sum(a * b for a, b in zip(self, escalar))
        return NotImplemented

    def __rmul__(self, escalar):
        """k * v (invocado cuando escalar.__mul__(v) retorna NotImplemented)"""
        return self.__mul__(escalar)

    def norma(self) -> float:
        """Norma euclidiana."""
        return sum(c ** 2 for c in self) ** 0.5


# ── Demo ──────────────────────────────────────────────────────────────────────
u = Vector(1, 2, 3)
v = Vector(4, 5, 6)

print("\n=== Vector ===")
print(f"u = {u}")
print(f"v = {v}")
print(f"u + v = {u + v}")
print(f"u - v = {u - v}")
print(f"u * 2 = {u * 2}")
print(f"3 * v = {3 * v}")
print(f"u · v (producto punto) = {u * v}")
print(f"||u|| = {u.norma():.4f}")
print(f"len(u) = {len(u)}")
print(f"3.0 in u: {3.0 in u}")
print(f"list(v) = {list(v)}")
print(f"\nDuck typing con primero_y_ultimo(Vector):")
print(f"  {primero_y_ultimo(v)}")

### 🔍 Análisis: `isinstance` vs. duck typing — cuándo usar cada uno

| Situación | Enfoque recomendado | Razón |
|-----------|--------------------|----|
| Función genérica que sólo llama métodos | Duck typing / EAFP | Máxima flexibilidad |
| Comparadores `__eq__`, `__add__` | `isinstance` → `NotImplemented` | Semántica correcta (Python hace retry) |
| Validación de API pública | `isinstance` de ABC | Mensaje de error claro |
| Ramificación por tipo (muchos `elif isinstance`) | Refactorizar — usar polimorfismo | Código frágil |

### El rol de `NotImplemented`

Cuando `__add__(self, other)` no sabe cómo sumar con `other`, debe retornar `NotImplemented` (no raise). Python entonces intenta `other.__radd__(self)`, dando oportunidad al otro objeto. Si ambos devuelven `NotImplemented`, Python lanza `TypeError` automáticamente.

---
## 🏛️ Sección 2 — Clases Base Abstractas (ABCs)

### Motivación

El duck typing es flexible, pero no garantiza que una clase **implemente todo lo necesario**. Un ABC formaliza un "contrato":

```text
Si tu clase hereda de MiABC, Python garantiza que implementaste
todos los @abstractmethod — o de lo contrario, no se puede instanciar.
```

### Sintaxis de un ABC

```python
from abc import ABC, abstractmethod

class FiguraGeometrica(ABC):       # hereda de ABC (o usa metaclass=ABCMeta)

    @abstractmethod
    def area(self) -> float:       # contrato OBLIGATORIO
        pass

    @abstractmethod
    def perimetro(self) -> float:
        pass

    def describir(self) -> str:    # método CONCRETO (mixin) — se hereda gratis
        return f"Área={self.area():.2f}, Perímetro={self.perimetro():.2f}"
```

```python
class Circulo(FiguraGeometrica):
    def __init__(self, radio):
        self._radio = radio
    def area(self):       return 3.14159 * self._radio ** 2
    def perimetro(self):  return 2 * 3.14159 * self._radio

# FiguraGeometrica()   -> TypeError: no se puede instanciar una clase abstracta
# Circulo(5)           -> OK: implementa todos los @abstractmethod
```

### La jerarquía ABC de Python (`collections.abc`)

`list`, `tuple`, `str` son **subclases** de `Sequence` de `collections.abc`. A su vez `Sequence` hereda de `Reversible` y `Collection`. Este árbol de ABCs es el fundamento de las estructuras de datos del lenguaje.

In [ ]:
from abc import ABC, abstractmethod
import math

# ─────────────────────────────────────────────────────────────────────────────
# ABC sencillo: FiguraGeometrica
# ─────────────────────────────────────────────────────────────────────────────

class FiguraGeometrica(ABC):
    """
    ABC para figuras en el plano.

    Contrato:
      - area()      (abstracto)
      - perimetro()  (abstracto)

    Heredado gratis:
      - describir()  (concreto / mixin)
      - __repr__     (concreto)
    """

    @abstractmethod
    def area(self) -> float:
        """Retorna el área de la figura."""

    @abstractmethod
    def perimetro(self) -> float:
        """Retorna el perímetro de la figura."""

    def describir(self) -> str:
        """Método concreto: las subclases lo heredan sin reimplementar."""
        return (f"{type(self).__name__}: "
                f"área={self.area():.4f}, perímetro={self.perimetro():.4f}")

    def __repr__(self):
        return self.describir()


class Circulo(FiguraGeometrica):
    def __init__(self, radio: float):
        if radio <= 0: raise ValueError("Radio debe ser positivo")
        self._radio = radio

    def area(self):       return math.pi * self._radio ** 2
    def perimetro(self):  return 2 * math.pi * self._radio


class Rectangulo(FiguraGeometrica):
    def __init__(self, ancho: float, alto: float):
        if ancho <= 0 or alto <= 0: raise ValueError("Dimensiones positivas")
        self._ancho = ancho
        self._alto  = alto

    def area(self):       return self._ancho * self._alto
    def perimetro(self):  return 2 * (self._ancho + self._alto)


class TrianguloRectangulo(FiguraGeometrica):
    def __init__(self, cateto_a: float, cateto_b: float):
        self._a = cateto_a
        self._b = cateto_b

    def area(self):       return 0.5 * self._a * self._b
    def perimetro(self):
        hipotenusa = (self._a**2 + self._b**2) ** 0.5
        return self._a + self._b + hipotenusa


# ── Demo ─────────────────────────────────────────────────────────────────────
figuras = [Circulo(5), Rectangulo(4, 6), TrianguloRectangulo(3, 4)]

print("=== Polimorfismo con FiguraGeometrica ===")
for f in figuras:
    print(f"  {f.describir()}")

print("\n=== isinstance con ABC ===")
for f in figuras:
    print(f"  {type(f).__name__:25s} -> isinstance(FiguraGeometrica): "
          f"{isinstance(f, FiguraGeometrica)}")

print("\n=== Ordenar por área (polimorfismo) ===")
por_area = sorted(figuras, key=lambda f: f.area())
for f in por_area:
    print(f"  área={f.area():.2f}  {type(f).__name__}")

print("\n=== Intentar instanciar ABC directamente ===")
try:
    FiguraGeometrica()
except TypeError as e:
    print(f"  TypeError esperado: {e}")

print("\n=== Clase incompleta (olvida implementar un método) ===")
class FiguraIncompleta(FiguraGeometrica):
    def area(self):      return 0.0
    # Falta perimetro()

try:
    FiguraIncompleta()
except TypeError as e:
    print(f"  TypeError esperado: {e}")

### 📖 El ABC `Secuencia` del libro (§2.7)

El ejemplo central de §2.7 es el ABC `Sequence` (renombrado `Secuencia` aquí). Muestra cómo los ABCs permiten implementar muchos métodos "gratis" con sólo dos métodos abstractos:

```text
Secuencia (ABC)
  Abstractos (deben implementarse):
    __len__()        <- cuántos elementos
    __getitem__(j)   <- elemento en posición j

  Concretos (heredados gratis, basados sólo en los dos anteriores):
    __contains__(val)   <- val in s
    __iter__()          <- for x in s
    index(val)          <- primera posición de val
    count(val)          <- cuántas veces aparece val
```

Este patrón es la esencia de las estructuras de datos: definir la **interfaz mínima** y construir el resto encima.

In [ ]:
from abc import ABC, abstractmethod

class Secuencia(ABC):
    """
    ABC inspirado en §2.7 del libro (Sequence ABC).

    Abstractos: __len__, __getitem__
    Concretos (mixin): __contains__, __iter__, index, count

    Cualquier clase que implemente sólo los dos métodos abstractos
    hereda automáticamente todo el comportamiento de secuencia.
    """

    @abstractmethod
    def __len__(self) -> int:
        """Número de elementos."""

    @abstractmethod
    def __getitem__(self, j: int):
        """Retorna el elemento en la posición j."""

    # ── Métodos concretos (mixin) — construidos sobre los abstractos ─────────
    def __contains__(self, val) -> bool:
        """Busca val linealmente. O(n)."""
        for j in range(len(self)):
            if self[j] == val:
                return True
        return False

    def __iter__(self):
        """Iterador que usa __len__ y __getitem__."""
        j = 0
        while j < len(self):
            yield self[j]
            j += 1

    def index(self, val) -> int:
        """Primera posición de val. Lanza ValueError si no está."""
        for j in range(len(self)):
            if self[j] == val:
                return j
        raise ValueError(f"{val!r} no está en la secuencia")

    def count(self, val) -> int:
        """Cuántas veces aparece val."""
        total = 0
        for j in range(len(self)):
            if self[j] == val:
                total += 1
        return total


# ── Implementación concreta 1: SecuenciaArray ────────────────────────────────
class SecuenciaArray(Secuencia):
    """Secuencia simple basada en una lista Python interna."""

    def __init__(self, *elementos):
        self._datos = list(elementos)

    def __len__(self):          return len(self._datos)
    def __getitem__(self, j):   return self._datos[j]

    def __repr__(self):
        return f"SecuenciaArray({self._datos})"


# ── Implementación concreta 2: Rango personalizado ───────────────────────────
class RangoPersonalizado(Secuencia):
    """
    Secuencia de enteros [inicio, fin) con paso, sin almacenarlos explícitamente.
    Equivalente conceptual al range() built-in.
    """

    def __init__(self, inicio: int, fin: int, paso: int = 1):
        if paso == 0:
            raise ValueError("El paso no puede ser cero")
        self._inicio = inicio
        self._fin    = fin
        self._paso   = paso
        # longitud calculada
        if paso > 0:
            self._longitud = max(0, (fin - inicio + paso - 1) // paso)
        else:
            self._longitud = max(0, (inicio - fin - paso - 1) // (-paso))

    def __len__(self):
        return self._longitud

    def __getitem__(self, j: int):
        if j < 0:
            j += self._longitud
        if not (0 <= j < self._longitud):
            raise IndexError("índice fuera de rango")
        return self._inicio + j * self._paso

    def __repr__(self):
        return f"Rango({self._inicio}, {self._fin}, paso={self._paso})"


# ── Demo ─────────────────────────────────────────────────────────────────────
sa = SecuenciaArray(10, 20, 30, 20, 40)
rp = RangoPersonalizado(0, 20, 3)   # 0, 3, 6, 9, 12, 15, 18

print("=== SecuenciaArray ===")
print(f"sa = {sa}")
print(f"len(sa) = {len(sa)}")
print(f"sa[2]   = {sa[2]}")
print(f"20 in sa = {20 in sa}")
print(f"99 in sa = {99 in sa}")
print(f"sa.index(30) = {sa.index(30)}")
print(f"sa.count(20) = {sa.count(20)}")
print(f"list(sa) = {list(sa)}")

print("\n=== RangoPersonalizado(0, 20, paso=3) ===")
print(f"rp = {rp}")
print(f"len(rp) = {len(rp)}")
print(f"rp[-1]  = {rp[-1]}")
print(f"9 in rp = {9 in rp}")
print(f"5 in rp = {5 in rp}")
print(f"list(rp) = {list(rp)}")

print("\n=== isinstance con Secuencia ===")
print(f"isinstance(sa, Secuencia):  {isinstance(sa, Secuencia)}")
print(f"isinstance(rp, Secuencia):  {isinstance(rp, Secuencia)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exploración: collections.abc — la jerarquía ABC estándar de Python
# ─────────────────────────────────────────────────────────────────────────────
import collections.abc as cabc

print("=== collections.abc — jerarquía parcial ===")

for nombre in ['Container', 'Iterable', 'Iterator', 'Sized',
               'Collection', 'Sequence', 'MutableSequence',
               'Mapping', 'MutableMapping', 'Set']:
    cls = getattr(cabc, nombre)
    abstractos = sorted(cls.__abstractmethods__) if hasattr(cls, '__abstractmethods__') else []
    print(f"  {nombre:20s} abstractos: {abstractos}")

print("\n=== list, str, tuple implementan Sequence ===")
for tipo in [list, tuple, str, range]:
    print(f"  {tipo.__name__:10s} es Sequence: {issubclass(tipo, cabc.Sequence)}")

print("\n=== dict es Mapping, no Sequence ===")
for tipo in [dict, set, frozenset]:
    is_seq = issubclass(tipo, cabc.Sequence)
    is_map = issubclass(tipo, cabc.Mapping)
    print(f"  {tipo.__name__:10s}  Sequence:{is_seq}  Mapping:{is_map}")

# ── Nuestra Secuencia vs. collections.abc.Sequence ────────────────────────────
print("\n=== Registro virtual (register) ===")
# collections.abc.Sequence.register(SecuenciaArray) permite que isinstance
# reconozca SecuenciaArray como Sequence sin herencia explícita
cabc.Sequence.register(SecuenciaArray)
print(f"Después de register: isinstance(sa, cabc.Sequence): "
      f"{isinstance(SecuenciaArray(1,2,3), cabc.Sequence)}")

---
## 🏗️ Sección 3 — Patrón de Diseño: Template Method

### Definición

> **Template Method**: la clase base define el *esqueleto* de un algoritmo en un método concreto, delegando ciertos pasos a subclases que los implementan.

Ya lo vimos en el Notebook 2 con `Progresion._avanzar()`. Ahora lo formalizamos con un ABC.

### Estructura general

```text
ClaseBase (ABC)                       <- define el esqueleto
  metodo_plantilla()                   <- concreto, llama a gancho()
  gancho() [abstractmethod]            <- las subclases implementan esto

SubclaseA(ClaseBase)                  <- especialización A
  def gancho(): ...

SubclaseB(ClaseBase)                  <- especialización B
  def gancho(): ...
```

### Beneficio clave

El código de orquestación (la parte difícil) vive **una sola vez** en la clase base. Las subclases sólo definen la variación que las hace diferentes.

In [ ]:
from abc import ABC, abstractmethod

# ─────────────────────────────────────────────────────────────────────────────
# Ejemplo 1: reportes con Template Method
# ─────────────────────────────────────────────────────────────────────────────

class GeneradorReporte(ABC):
    """
    Template Method para generar reportes en distintos formatos.
    
    El método reporte() es el template — define el esqueleto.
    Las subclases sólo implementan los "ganchos":
      _encabezado(), _cuerpo(datos), _pie()
    """

    def reporte(self, titulo: str, datos: list) -> str:
        """Método plantilla — NO se sobreescribe."""
        partes = [
            self._encabezado(titulo),
            self._cuerpo(datos),
            self._pie()
        ]
        return '\n'.join(p for p in partes if p)

    @abstractmethod
    def _encabezado(self, titulo: str) -> str:
        pass

    @abstractmethod
    def _cuerpo(self, datos: list) -> str:
        pass

    def _pie(self) -> str:
        """Gancho opcional — tiene implementación por defecto."""
        return ""


class ReporteTexto(GeneradorReporte):
    def _encabezado(self, titulo):
        sep = "=" * (len(titulo) + 4)
        return f"{sep}\n  {titulo}\n{sep}"

    def _cuerpo(self, datos):
        return '\n'.join(f"  - {d}" for d in datos)

    def _pie(self):
        return f"\n  Total de elementos: {self._conteo}"

    def reporte(self, titulo, datos):
        self._conteo = len(datos)
        return super().reporte(titulo, datos)


class ReporteHTML(GeneradorReporte):
    def _encabezado(self, titulo):
        return f"<h1>{titulo}</h1>"

    def _cuerpo(self, datos):
        items = '\n'.join(f"  <li>{d}</li>" for d in datos)
        return f"<ul>\n{items}\n</ul>"

    def _pie(self):
        return "<footer>Generado automáticamente</footer>"


class ReporteCSV(GeneradorReporte):
    def __init__(self, separador=','):
        self._sep = separador

    def _encabezado(self, titulo):
        return f"# {titulo}"

    def _cuerpo(self, datos):
        return '\n'.join(str(d) for d in datos)


# ── Demo ─────────────────────────────────────────────────────────────────────
datos = ["Python 3.12", "Django 5.0", "FastAPI 0.111", "SQLAlchemy 2.0"]

for generador in [ReporteTexto(), ReporteHTML(), ReporteCSV()]:
    print(f"\n{'─'*50}")
    print(f"  Tipo: {type(generador).__name__}")
    print('─'*50)
    print(generador.reporte("Librerías Python", datos))


# ─────────────────────────────────────────────────────────────────────────────
# Ejemplo 2: Progresion como Template Method formal (ABC)
# ─────────────────────────────────────────────────────────────────────────────

class ProgresionABC(ABC):
    """Versión ABC estricta de la clase Progresion del libro."""

    def __init__(self, inicio=0):
        self._actual = inicio

    @abstractmethod
    def _avanzar(self) -> None:
        """Actualiza self._actual al siguiente valor de la progresión."""

    def __next__(self):
        if self._actual is None:
            raise StopIteration
        resp = self._actual
        self._avanzar()
        return resp

    def __iter__(self):
        return self

    def tomar(self, n: int) -> list:
        return [next(self) for _ in range(n)]


class ProgAritABC(ProgresionABC):
    def __init__(self, incremento=1, inicio=0):
        super().__init__(inicio)
        self._inc = incremento

    def _avanzar(self):
        self._actual += self._inc

print(f"\n\nProgresion ABC: {ProgAritABC(5).tomar(6)}")

---
## 🧪 Tests Unitarios

In [ ]:
import unittest

class TestVector(unittest.TestCase):

    def test_suma(self):
        self.assertEqual(Vector(1, 2) + Vector(3, 4), Vector(4, 6))

    def test_resta(self):
        self.assertEqual(Vector(5, 5) - Vector(2, 3), Vector(3, 2))

    def test_escalar_derecho(self):
        self.assertEqual(Vector(1, 2, 3) * 2, Vector(2, 4, 6))

    def test_escalar_izquierdo(self):
        self.assertEqual(3 * Vector(1, 0), Vector(3, 0))

    def test_producto_punto(self):
        self.assertAlmostEqual(Vector(1, 2, 3) * Vector(4, 5, 6), 32.0)

    def test_norma(self):
        self.assertAlmostEqual(Vector(3, 4).norma(), 5.0)

    def test_contains(self):
        v = Vector(1, 2, 3)
        self.assertIn(2.0, v)
        self.assertNotIn(9.0, v)

    def test_iter(self):
        self.assertEqual(list(Vector(7, 8, 9)), [7.0, 8.0, 9.0])


class TestSecuencia(unittest.TestCase):

    def setUp(self):
        self.sa = SecuenciaArray(10, 20, 30, 20, 40)
        self.rp = RangoPersonalizado(0, 10, 2)   # 0, 2, 4, 6, 8

    def test_len(self):
        self.assertEqual(len(self.sa), 5)
        self.assertEqual(len(self.rp), 5)

    def test_getitem(self):
        self.assertEqual(self.sa[0], 10)
        self.assertEqual(self.sa[-1], 40)
        self.assertEqual(self.rp[3], 6)

    def test_contains(self):
        self.assertIn(20, self.sa)
        self.assertNotIn(99, self.sa)
        self.assertIn(4, self.rp)

    def test_iter(self):
        self.assertEqual(list(self.sa), [10, 20, 30, 20, 40])
        self.assertEqual(list(self.rp), [0, 2, 4, 6, 8])

    def test_index(self):
        self.assertEqual(self.sa.index(30), 2)
        with self.assertRaises(ValueError):
            self.sa.index(99)

    def test_count(self):
        self.assertEqual(self.sa.count(20), 2)
        self.assertEqual(self.sa.count(10), 1)
        self.assertEqual(self.sa.count(99), 0)

    def test_isinstance_abc(self):
        self.assertIsInstance(self.sa, Secuencia)
        self.assertIsInstance(self.rp, Secuencia)


class TestFiguras(unittest.TestCase):

    def test_circulo_area(self):
        c = Circulo(5)
        self.assertAlmostEqual(c.area(), math.pi * 25)

    def test_rectangulo_perimetro(self):
        r = Rectangulo(3, 4)
        self.assertAlmostEqual(r.perimetro(), 14.0)

    def test_triangulo_area(self):
        t = TrianguloRectangulo(3, 4)
        self.assertAlmostEqual(t.area(), 6.0)

    def test_abstract_no_instanciable(self):
        with self.assertRaises(TypeError):
            FiguraGeometrica()

    def test_polimorfismo_describir(self):
        figuras = [Circulo(1), Rectangulo(2, 3), TrianguloRectangulo(3, 4)]
        for f in figuras:
            desc = f.describir()
            self.assertIn('área', desc)
            self.assertIn('perímetro', desc)


# ── Run ───────────────────────────────────────────────────────────────────────
loader = unittest.TestLoader()
suite  = unittest.TestSuite()
for cls in [TestVector, TestSecuencia, TestFiguras]:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tabla comparativa: duck typing vs. isinstance vs. ABC
# ─────────────────────────────────────────────────────────────────────────────

comparacion = [
    ("Duck typing",  "Flexible",  "Sin garantías en tiempo de definición", "Función genérica"),
    ("isinstance",   "Explícito", "Lógica rígida si hay muchos tipos",     "Operadores (__eq__)"),
    ("ABC",          "Garantía",  "Más estructura, menos flexibilidad",    "API pública formal"),
]

print(f"{'Enfoque':15s} {'Ventaja':20s} {'Limitación':40s} {'Cuándo usarlo':25s}")
print("-" * 105)
for enfoque, ventaja, limitacion, cuando in comparacion:
    print(f"{enfoque:15s} {ventaja:20s} {limitacion:40s} {cuando:25s}")

print("\n=== Verificación de abstract methods ===")
import inspect
print(f"FiguraGeometrica.__abstractmethods__: {FiguraGeometrica.__abstractmethods__}")
print(f"Secuencia.__abstractmethods__:        {Secuencia.__abstractmethods__}")
print(f"GeneradorReporte.__abstractmethods__: {GeneradorReporte.__abstractmethods__}")

# ABCs en collections.abc con @abstractproperty
print("\n=== Ejemplo @property abstracta ===")

class ContenedorOrdenado(ABC):
    
    @property
    @abstractmethod
    def maximo(self):
        """Elemento máximo del contenedor."""
    
    @property
    @abstractmethod
    def minimo(self):
        """Elemento mínimo del contenedor."""

class ListaOrdenada2(ContenedorOrdenado):
    def __init__(self, *items):
        self._items = sorted(items)
    
    @property
    def maximo(self): return self._items[-1]
    
    @property
    def minimo(self): return self._items[0]

lo = ListaOrdenada2(5, 2, 8, 1, 9)
print(f"ListaOrdenada2(5,2,8,1,9) — mín={lo.minimo}, máx={lo.maximo}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Gráfico 1: área de figuras geométricas ────────────────────────────────────
ax = axes[0]
radios_circulo = [1, 2, 3, 4, 5]
areas_c = [Circulo(r).area()         for r in radios_circulo]
areas_r = [Rectangulo(r, r).area()   for r in radios_circulo]
areas_t = [TrianguloRectangulo(r, r).area() for r in radios_circulo]

ax.plot(radios_circulo, areas_c, 'b-o', label='Círculo (radio=r)', markersize=6)
ax.plot(radios_circulo, areas_r, 'r-s', label='Cuadrado (lado=r)',  markersize=6)
ax.plot(radios_circulo, areas_t, 'g-D', label='TriRectángulo (a=b=r)', markersize=6)
ax.set_xlabel('Parámetro r')
ax.set_ylabel('Área')
ax.set_title('Polimorfismo: FiguraGeometrica.area()')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Gráfico 2: jerarquía ABC ──────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Patrón ABC — estructura', fontsize=12, fontweight='bold')

def caja(ax, cx, cy, w, h, texto, color, fontsize=8.5):
    rect = mpatches.FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                                    boxstyle='round,pad=0.15',
                                    facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(cx, cy, texto, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def flecha(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.5, mutation_scale=14))

# ABC
caja(ax2, 5, 8.5, 4.5, 1.1, 'FiguraGeometrica (ABC)\narea() @abstract  perimetro() @abstract\ndescribir() concreto', '#AED6F1', fontsize=7)

# Concretas
for cx, nom, color in [(2, 'Circulo', '#A9DFBF'), (5, 'Rectangulo', '#F9E79F'), (8, 'Triangulo', '#FAD7A0')]:
    caja(ax2, cx, 6.5, 2.8, 0.9, nom, color, fontsize=8)
    flecha(ax2, cx, 6.95, 5, 7.95)

# descripciones
ax2.text(5, 5.7, 'Cada subclase implementa area() y perimetro()', ha='center', fontsize=8, style='italic')
ax2.text(5, 5.2, 'Todas heredan describir() sin reimplementar', ha='center', fontsize=8, color='#555')

# Secuencia ABC
caja(ax2, 5, 3.9, 4.5, 1.3,
     'Secuencia (ABC)\n__len__ @abstract  __getitem__ @abstract\n__contains__, __iter__, index, count – concretos',
     '#D7BDE2', fontsize=6.5)

caja(ax2, 2.5, 2.2, 2.8, 0.9, 'SecuenciaArray', '#FADBD8', fontsize=8)
caja(ax2, 7.5, 2.2, 2.8, 0.9, 'RangoPersonalizado', '#D5F5E3', fontsize=8)
flecha(ax2, 2.5, 2.65, 5, 3.25)
flecha(ax2, 7.5, 2.65, 5, 3.25)

fig.tight_layout(pad=2)
plt.suptitle('Cap. 2 — §2.7 Polimorfismo y ABCs', y=1.01, fontsize=10, style='italic')
plt.savefig('polimorfismo_abcs.png', bbox_inches='tight', dpi=100)
plt.show()
print("Gráfico guardado.")

---
## 📋 Resumen del Notebook

### Conceptos clave del Capítulo §2.7

| Concepto | Descripción | Ejemplo |
|----------|-------------|---------|
| **Polimorfismo** | Misma operación, distintos tipos | `area()` en `Circulo`, `Rectangulo`, `Triangulo` |
| **Duck typing** | Quién importa es lo que hace, no su clase | `len(x)`, `for v in x` |
| **ABC** | Contrato formal de implementación | `class F(ABC): @abstractmethod def area():` |
| **`@abstractmethod`** | Método que las subclases DEBEN implementar | Impide instanciar la clase base |
| **Métodos mixin** | Concretos en el ABC, basados en los abstractos | `__contains__` → usa `__len__` + `__getitem__` |
| **Template Method** | Esqueleto en la base, pasos en las subclases | `GeneradorReporte.reporte()` |
| **`collections.abc`** | Jerarquía ABC estándar de Python | `Sequence`, `Mapping`, `Iterator`... |

### Síntesis del Capítulo 2

El capítulo 2 completo construye una progresión conceptual:

```text
§2.1        → Principios OOP: encapsulamiento, abstracción, polimorfismo, herencia
§2.2-2.3    → Clases en Python: __init__, @property, métodos especiales
§2.4-2.5    → Herencia: super(), MRO, namespaces, name mangling
§2.7        → ABCs: contratos formales, duck typing, Template Method
```

---
## ✅ Autoevaluación

- [ ] Puedo explicar por qué Python prefiere duck typing sobre verificación de tipos explícita
- [ ] Sé crear un ABC con `@abstractmethod` y métodos concretos (mixin)
- [ ] Implementé `Secuencia` con `__len__`/`__getitem__` y heredé `__iter__`, `__contains__`, `index`
- [ ] Entiendo el patrón Template Method y puedo dar un ejemplo propio
- [ ] Sé qué hay en `collections.abc` y por qué `list` es una `Sequence`

## 📚 Referencias y Conexiones

### Fuentes del capítulo
- Goodrich, Tamassia & Goldwasser — Cap. 2:
  - §2.7 Abstract Base Classes
  - Apéndice: `collections.abc` hierarchy
- [Python `abc` module reference](https://docs.python.org/3/library/abc.html)
- [collections.abc — ABCs for containers](https://docs.python.org/3/library/collections.abc.html)
- [PEP 3119 — ABC](https://peps.python.org/pep-3119/)

### Conexiones con el resto del texto

| Concepto | Capítulo | Aplicación |
|----------|---------|-----------|
| `Secuencia` ABC | Cap. 5 — Array-Based | Base para `ArrayStack`, `ArrayQueue` |
| `Progresion` / Template Method | Cap. 7 — Linked Lists | `LinkedList` base con nodos abstractos |
| Duck typing en algoritmos | Cap. 12 — Sorting | `sorted()` sólo necesita `__lt__` |
| ABCs formales | Cap. 10 — Maps | `MutableMapping` ABC |

---
© 2026 Cátedra Programación III — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).